# 04 — QLoRA Fine-Tuning

Load the same base model in 4-bit NF4 and attach LoRA adapters. Reuse the dataset and shared configuration from the LoRA experiment.

In [ ]:
import os, json, time, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, TaskType
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "../models/qlora_adapter"

## Quantization configuration

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
)

## Load quantized model

In [ ]:
def load_4bit_base(model_id=MODEL_ID):
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA 4-bit training requires a compatible CUDA environment.")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = prepare_model_for_kbit_training(model)
    return model, tokenizer

# model, tokenizer = load_4bit_base()

## Adapter setup

In [ ]:
qlora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
# model = get_peft_model(model, qlora_config)
# model.print_trainable_parameters()

## Memory-aware training options

In [ ]:
qlora_training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=100,
    save_strategy="steps",
    optim="paged_adamw_8bit",
    report_to="none",
    fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    seed=42,
)

## Dataset and trainer

Use the exact same prepared splits and formatting function as Notebook 03.

In [ ]:
# train_dataset = load_jsonl_dataset("../data/train.jsonl").map(format_training_record)
# validation_dataset = load_jsonl_dataset("../data/validation.jsonl").map(format_training_record)
# trainer = SFTTrainer(
#     model=model,
#     args=qlora_training_args,
#     train_dataset=train_dataset,
#     eval_dataset=validation_dataset,
#     processing_class=tokenizer,
#     dataset_text_field="text",
#     max_seq_length=512,
# )

## Train and capture measurements

In [ ]:
# torch.cuda.reset_peak_memory_stats()
# start = time.perf_counter()
# result = trainer.train()
# training_seconds = time.perf_counter() - start
# peak_allocated = torch.cuda.max_memory_allocated() / 1024**3
# peak_reserved = torch.cuda.max_memory_reserved() / 1024**3
# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)
# metrics = {
#     "training_seconds": training_seconds,
#     "peak_allocated_gb": peak_allocated,
#     "peak_reserved_gb": peak_reserved,
#     **result.metrics,
# }
# with open("../results/qlora_training_metrics.json", "w") as f:
#     json.dump(metrics, f, indent=2)
# print(metrics)

## Adapter integrity check

In [ ]:
# adapter_path = Path(OUTPUT_DIR)
# assert adapter_path.exists(), "Adapter directory not created."
# adapter_weights = list(adapter_path.glob("*.safetensors")) + list(adapter_path.glob("*.bin"))
# assert adapter_weights, "No adapter weight file found."
# print("Adapter checkpoint files:", [p.name for p in adapter_weights])

## Notes

QLoRA changes memory requirements by quantizing frozen base weights. Adapter settings, dataset, prompt format, and training schedule should remain matched to the LoRA comparison where feasible.

## Verified QLoRA configuration and runtime audit

The following outputs are generated from the notebook's declared settings and the current execution environment. They are not training results.

In [1]:
qlora_settings = {
    "base_model": MODEL_ID,
    "output_dir": OUTPUT_DIR,
    "quantization": "4-bit",
    "quant_type": "nf4",
    "double_quantization": True,
    "adapter_rank": 16,
    "adapter_alpha": 32,
    "adapter_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "learning_rate": 2e-4,
    "epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "max_sequence_length": 512,
}
for key, value in qlora_settings.items():
    print(f"{key}: {value}")


base_model: Qwen/Qwen2.5-3B-Instruct
output_dir: ../models/qlora_adapter
quantization: 4-bit
quant_type: nf4
double_quantization: True
adapter_rank: 16
adapter_alpha: 32
adapter_dropout: 0.05
target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj']
learning_rate: 0.0002
epochs: 1
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
max_sequence_length: 512

## Verified runtime readiness

The model-loading cell intentionally stops without a compatible CUDA runtime. This check records the current environment so the experiment can be resumed on a GPU host.

In [2]:
import importlib.util, torch, platform
required = ["transformers", "datasets", "peft", "trl", "bitsandbytes"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Missing QLoRA dependencies:", missing)
if not torch.cuda.is_available() or missing:
    print("Readiness: BLOCKED — install dependencies and select a supported CUDA GPU runtime.")
else:
    print("Readiness: dependency and CUDA checks passed; verify VRAM before loading the model.")


Python: 3.13.5
PyTorch: 2.10.0+cpu
CUDA available: False
GPU: No CUDA GPU available
CPU logical cores: 5
System RAM total (GB): 5.81
Dependencies:
  torch: installed
  transformers: not installed
  datasets: not installed
  peft: not installed
  trl: not installed
  bitsandbytes: not installed
  pandas: installed
  psutil: installed
Readiness: BLOCKED — install dependencies and select a supported CUDA GPU runtime.

## Training result status

No fabricated loss, throughput, GPU memory, or adapter-size outputs are included. The training cell is commented out in the supplied notebook; execute it on a configured GPU to generate those results.